# The birthday paradox, and hash-table collisions

With $d$ equally likely birthdays and $n$ people, the probability that all $n$ are distinct is

$$P(\text{all distinct}) = \prod_{i=0}^{n-1}\frac{d-i}{d} = \frac{d!}{d^{\,n}\,(d-n)!},$$

so the probability that at least two coincide is

$$P(\text{collision}) = 1 - \prod_{i=0}^{n-1}\frac{d-i}{d}.$$

For $d = 365$ this crosses $\tfrac{1}{2}$ at just $n = 23$ — far fewer than intuition suggests, because what matters is the number of *pairs*, $\binom{n}{2} = \tfrac{n(n-1)}{2}$, which grows quadratically in $n$.

A hash table is the same problem with $d = m$: dropping $n$ keys into $m$ buckets, the expected number of colliding pairs is $\binom{n}{2}/m$. Collisions therefore appear long before the table is full.

## A minimal hash map

Separate chaining: $m$ buckets, each a list. A universal hash spreads keys evenly,

$$h(k) = \big((a\,k + b) \bmod p\big) \bmod m,$$

with $a, b$ chosen at random and $p$ the Mersenne prime $2^{61}-1$. When the load factor $n/m$ passes `MAX_LOAD = 0.75`, the table grows to the next prime in the list and every key is rehashed — which is why the bucket counts are a *list* of primes, not a single number.

In [1]:
import random

PRIMES = [3, 7, 13, 31, 61, 127, 251, 509, 1021, 2039, 4093, 8191]
MERSENNE = (1 << 61) - 1
MAX_LOAD = 0.75


class HashMap:
    def __init__(self, n_buckets=PRIMES[0], grow=True):
        self.m = n_buckets
        self.grow = grow
        self.n = 0
        self.buckets = [[] for _ in range(self.m)]
        self.a = random.randrange(1, MERSENNE)
        self.b = random.randrange(0, MERSENNE)

    def _index(self, key):
        return ((self.a * key + self.b) % MERSENNE) % self.m

    def put(self, key, value):
        chain = self.buckets[self._index(key)]
        for i, (k, _) in enumerate(chain):
            if k == key:
                chain[i] = (key, value)
                return
        chain.append((key, value))
        self.n += 1
        if self.grow and self.n > MAX_LOAD * self.m:
            self._resize()

    def get(self, key):
        for k, v in self.buckets[self._index(key)]:
            if k == key:
                return v
        raise KeyError(key)

    def _resize(self):
        bigger = next((p for p in PRIMES if p > self.m), None)
        if bigger is None:
            return
        self.m = bigger
        items = [pair for chain in self.buckets for pair in chain]
        self.buckets = [[] for _ in range(self.m)]
        for k, v in items:
            self.buckets[self._index(k)].append((k, v))

## Collisions vs. load factor

The map grows to keep its load factor low — but that hides what collisions actually look like. So here growth is turned off (`grow=False`) with $m = 251$ buckets fixed, and $n = \alpha m$ random keys are inserted through the same hash. The chart counts how many buckets end up holding $0, 1, 2, \dots$ keys. Drag the slider to raise the load factor $\alpha$: buckets with two or more keys — collisions — grow quadratically, exactly as $\binom{n}{2}$ predicts, which is why the real map resizes at $\alpha = 0.75$ rather than letting $\alpha$ climb.

In [3]:
import plotly.graph_objects as go

random.seed(0)
m = PRIMES[6]
load_factors = [round(0.25 * t, 2) for t in range(1, 17)]
max_len = 9

fig = go.Figure()
for alpha in load_factors:
    hm = HashMap(m, grow=False)
    n = int(round(alpha * m))
    for key in random.sample(range(1_000_000), n):
        hm.put(key, None)
    sizes = [len(bucket) for bucket in hm.buckets]
    counts = [sizes.count(j) for j in range(max_len)]
    counts.append(sum(s >= max_len for s in sizes))
    fig.add_trace(go.Bar(x=list(range(max_len + 1)), y=counts,
                         visible=(alpha == 1.0), name=f"alpha={alpha}"))

steps = [
    dict(method="update",
         args=[{"visible": [j == idx for j in range(len(load_factors))]},
               {"title": f"Bucket occupancy at load factor alpha = {alpha}"}],
         label=str(alpha))
    for idx, alpha in enumerate(load_factors)
]
fig.update_layout(
    sliders=[dict(active=load_factors.index(1.0),
                  currentvalue={"prefix": "alpha = "}, steps=steps)],
    title="Bucket occupancy at load factor alpha = 1.0",
    xaxis_title="keys in a bucket (9 = 9 or more)",
    yaxis_title="number of buckets",
    bargap=0.1,
    margin=dict(t=60),
)
fig.show()

**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-birthday-paradox-and-hashing.ipynb)**